# 12 · Production Concerns: Limits, Failures, Batches and Testing

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama for §2–4; §5 runs offline.

A crew that works in a notebook still has to survive production: stuck agents,
slow tools, provider limits, batches of requests, and prompt changes that
silently break behaviour. This notebook tests how CrewAI's limits **actually
behave** (one of them in a way you wouldn't guess), handles failures at the entry
point, and shows how to test crews and flows with a **fake LLM**.

**You will learn**
- `max_iter`: why hitting the limit gives you a **guess**, not an error
- `max_execution_time`: timeouts that fire late, and why
- Rate limits (`max_rpm`), model timeouts, and batching with `kickoff_for_each_async`
- An entry point that catches every failure mode from this section
- Testing flows and crews with a scripted `BaseLLM`
- A production checklist

**Prerequisites:** notebooks 01–11.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import time, asyncio
from crewai import Agent, Task, Crew, BaseLLM
from crewai.tools import tool

## 1 · Why: limits must fail loudly

A limit is only useful if you know when it was hit. The dangerous kind of limit
lets the run *finish anyway* with an output that looks normal. Let's check
CrewAI's.

## 2 · `max_iter`: a forced answer, not an error

`max_iter` (default 25) caps how many reasoning/tool steps an agent may take on a
task. What happens when a task genuinely needs more? A counter that must call a
tool ten times, one call at a time, with `max_iter=3`:

In [ ]:
calls = []

@tool('Next step')
def next_step(n: int) -> int:
    """Return the number after n."""
    calls.append(n)
    return n + 1

counter = Agent(role='Counter', goal='Count exactly', backstory='Calls Next step one at a time, never skips.',
                llm=get_llm(), tools=[next_step], max_iter=3)
count_task = Task(description='Start at 0 and call Next step repeatedly, one call at a time, until the result is 10. '
                              'Report the final number.', expected_output='The final number', agent=counter)

out = await Crew(agents=[counter], tasks=[count_task]).kickoff_async()
print('answer        :', repr(out.raw))
print('tool calls made:', len(calls), '→ numbers passed in:', calls)

In our runs the crew **finished normally** with the answer **"10"**, after only a
few tool calls, far too few to have counted to 10. When an agent reaches
`max_iter`, CrewAI doesn't raise. It asks the model to **give its final answer
now**, and the model gave the answer it knew was expected.

That makes `max_iter` a cost guard, **not** a correctness guard. If hitting it
matters (and it usually does), detect it yourself: count tool calls (as above, or
with an event listener from notebook 10), or validate the output with a guardrail
(notebook 07).

## 3 · `max_execution_time`: timeouts that fire late

`max_execution_time=seconds` on an agent limits how long a task may run. A tool
that hangs for 6 seconds, with a 3-second limit:

In [ ]:
@tool('Slow API')
def slow_api(query: str) -> str:
    """Query a slow external API."""
    time.sleep(6)                                      # a hung dependency
    return 'late result'

caller = Agent(role='Caller', goal='Query APIs', backstory='Uses the tool.', llm=get_llm(),
               tools=[slow_api], max_execution_time=3)
t0 = time.perf_counter()
try:
    await Crew(agents=[caller], tasks=[Task(description='Use the Slow API tool with query "x" and report the result.',
                                            expected_output='The result', agent=caller)]).kickoff_async()
except TimeoutError as err:
    print(f'TimeoutError after {time.perf_counter() - t0:.1f}s: {str(err)[:100]}…')

The limit **did** fire (a `TimeoutError`), but after about **8 seconds, not 3**.
The tool was a synchronous function that couldn't be interrupted, so the timeout
could only take effect once it returned. The same lesson appeared in every
section: **put timeouts on the calls that can hang** (the HTTP client inside the
tool), and treat task-level timeouts as a backstop.

## 4 · Throughput: rate limits, model timeouts, batches

- **`max_rpm`** on a crew or agent caps requests per minute, so a crew waits instead
  of hitting a provider's rate limit.
- **`LLM(timeout=...)`** sets the timeout for each model request.
- **Batches.** `crew.kickoff_for_each_async(inputs=[...])` runs the same crew for
  many inputs concurrently and returns one output per input:

In [ ]:
explainer = Agent(role='Explainer', goal='Define terms', backstory='One sentence each.', llm=get_llm())
glossary = Crew(agents=[explainer], tasks=[Task(description='Define {term} in one sentence.',
                                                 expected_output='One sentence', agent=explainer)])

t0 = time.perf_counter()
outputs = await glossary.kickoff_for_each_async(inputs=[{'term': t} for t in ('latency', 'throughput', 'jitter')])
print(f'{len(outputs)} results in {time.perf_counter() - t0:.1f}s')
for o in outputs:
    print(' •', o.raw)

Keep batches bounded (and `max_rpm` set) when calling a paid or rate-limited API;
a list of 1,000 inputs starts 1,000 crews.

## 5 · An entry point that handles every failure

Everything this section found, in one place. Each failure mode becomes a
deliberate response instead of a stack trace:

```python
async def run_request(inputs):
    try:
        out = await crew.kickoff_async(inputs=inputs)
    except TimeoutError:                        # max_execution_time (notebook 12)
        return 'This took too long; please try again.'
    except ToolExecutionFailedError:            # tool_failure_policy='raise' (notebook 04)
        return 'A system we depend on failed.'
    except Exception as err:                    # guardrail retries exhausted: a plain Exception (notebook 07)
        return f'Could not produce a valid answer: {err}'
    if out.has_tool_failures:                   # recorded tool failures (notebook 04)
        log_for_review(out)
    return out.raw
```

## 6 · Testing with a fake LLM

LLM output varies, so test **your logic** (routing, fallbacks, guardrails, flow
structure) with a model that returns scripted answers. CrewAI's `BaseLLM` needs
only a `call` method:

In [ ]:
class ScriptedLLM(BaseLLM):
    # Returns scripted replies in order: deterministic, instant, free.
    def __init__(self, replies):
        super().__init__(model='scripted')
        self._replies = list(replies)

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None, **kwargs):
        return self._replies.pop(0)

**Testing a flow's routing.** Build the flow from a factory that takes the LLM,
exactly the pattern from the other sections. The test scripts the classifier's
answer and checks which branch ran:

In [ ]:
from pydantic import BaseModel
from crewai.flow import Flow, start, router, listen


class TicketState(BaseModel):
    text: str = ''
    route: str = ''


def build_triage(llm):
    class Triage(Flow[TicketState]):
        @start()
        def classify(self):
            answer = llm.call(f'One word, billing or technical: {self.state.text}').strip().lower()
            self.state.route = answer if answer in ('billing', 'technical') else 'human'   # safe default

        @router(classify)
        def decide(self):
            return self.state.route

        @listen('billing')
        def handle_billing(self):  return 'refund desk'

        @listen('technical')
        def handle_technical(self): return 'support engineer'

        @listen('human')
        def handle_human(self):     return 'human queue'
    return Triage(suppress_flow_events=True)


async def test_billing():
    assert await build_triage(ScriptedLLM(['billing'])).kickoff_async() == 'refund desk'

async def test_rambling_goes_to_a_human():
    assert await build_triage(ScriptedLLM(['Hmm, maybe billing?'])).kickoff_async() == 'human queue'

for test in (test_billing, test_rambling_goes_to_a_human):
    await test()
    print('✅', test.__name__)

**Testing a crew with the fake LLM.** A crew's agents can use the scripted model
too, so you can test the wiring, the context passing, and guardrails without a
real model:

In [ ]:
from crewai import TaskOutput

def max_10_words(output: TaskOutput):
    ok = len(output.raw.split()) <= 10
    return (True, output.raw) if ok else (False, 'Too long, max 10 words.')


async def test_guardrail_retries_until_valid():
    fake = ScriptedLLM(['This answer is far too long to pass the ten word limit we set here.',
                        'Short answer.'])
    agent = Agent(role='Writer', goal='Write', backstory='Brief.', llm=fake)
    task = Task(description='Say something short.', expected_output='Short text', agent=agent,
                guardrail=max_10_words, guardrail_max_retries=2)
    out = await Crew(agents=[agent], tasks=[task]).kickoff_async()
    assert out.raw == 'Short answer.', out.raw

await test_guardrail_retries_until_valid()
print('✅ test_guardrail_retries_until_valid')

These tests run in milliseconds with no model at all. They catch the regressions
that matter most (a route that no longer triggers, a guardrail that stopped
retrying, a fallback that disappeared) on every change. Put them in `tests/`
and run them with `pytest`. Measure answer **quality** separately, with
evaluations on real models.

## 7 · Production checklist

- [ ] `await crew.kickoff_async()` everywhere async (servers, notebooks)
- [ ] An entry point that catches timeouts, tool failures and guardrail exhaustion (§5)
- [ ] `max_iter` treated as a **cost** limit; completion checked separately (tool counts, guardrails)
- [ ] Timeouts inside tools on anything that can hang; `max_execution_time` as a backstop
- [ ] `max_rpm` and bounded batches for rate-limited providers
- [ ] Guardrails on tasks whose outputs others build on, including the leaked-tool-call check (notebook 11)
- [ ] Tool failures checked (`has_tool_failures`) before trusting an answer
- [ ] Flows with `@persist` and **idempotent** steps for anything that must survive a crash (notebook 09)
- [ ] Human approval in a Flow before irreversible actions (notebook 08)
- [ ] `BaseTool` classes if you use crew checkpoints, and a tested restore
- [ ] MCP tools via `MCPServerAdapter`, filtered per agent; verify agents actually have them
- [ ] Telemetry and tracing configured deliberately
- [ ] Token usage measured per run (fresh LLM objects, notebook 02)
- [ ] Route/flow/guardrail tests with a scripted LLM, in CI

## 8 · Check yourself

<details><summary><b>1.</b> An agent hits <code>max_iter</code>. Do you get an error?</summary>

No. CrewAI forces a final answer, so the crew completes with an answer that may be a guess (we got "10" without counting). Treat `max_iter` as a cost limit and verify completion separately.
</details>

<details><summary><b>2.</b> Why did a 3-second <code>max_execution_time</code> fire after about 8 seconds?</summary>

The hung tool was synchronous and couldn't be interrupted, so the timeout took effect only when it returned. Put timeouts inside the tool on the call that can hang.
</details>

<details><summary><b>3.</b> What exception arrives when a guardrail's retries run out, and why does that matter for your entry point?</summary>

A plain `Exception`, not a specific type. Your entry point needs a general `except` (after the specific ones) to turn it into a deliberate response.
</details>

<details><summary><b>4.</b> How do you test a flow's routing without a model?</summary>

Build the flow from a factory that takes the LLM, pass a scripted `BaseLLM` subclass that returns chosen answers, and assert which branch ran. It's fast, free and deterministic.
</details>

## Takeaway

Know how each limit fails: `max_iter` **forces a guess** instead of raising, and
`max_execution_time` fires only when a blocking call returns. Guard completion
yourself, put timeouts where calls can hang, bound batches, and funnel every
failure through one entry point. Build crews and flows from factories so a
scripted `BaseLLM` can test your logic on every commit.

That completes the teaching notebooks. Next → `project.ipynb`: the Nimbus
Briefing Assistant as a CrewAI Flow.